# Colab End-to-End: Hindi Punctuation, Ordering, Grouping & Translation

This notebook runs the project pipeline in Google Colab.

**Constraints:** No GPT/Gemini/Claude. Keep GPU memory < 1.5 GB if using GPU (CPU is recommended for compliance in this pipeline).

**IndicTrans2:** requires accepting model access on Hugging Face and providing `HF_TOKEN` in Colab secrets. If unavailable, translation falls back to MarianMT (`Helsinki-NLP/opus-mt-hi-en`).

## 1) Runtime setup (recommended: CPU for <1.5GB GPU constraint)
In Colab: `Runtime > Change runtime type` → select **CPU** (or GPU only if you will measure and stay under 1.5 GB VRAM).

In [4]:
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/gitanshu-chavda/Punctuation_Adder-and-Sentence-Ordering.git"
BRANCH = "sen-ord-1"
WORKDIR = pathlib.Path('/content/Punctuation_Adder-and-Sentence-Ordering')

if not WORKDIR.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(WORKDIR)])
else:
    subprocess.check_call(['git', '-C', str(WORKDIR), 'fetch', 'origin', BRANCH])
    subprocess.check_call(['git', '-C', str(WORKDIR), 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', str(WORKDIR), 'pull', '--ff-only', 'origin', BRANCH])

os.chdir(WORKDIR)
print('CWD:', os.getcwd())
print('Branch:', subprocess.check_output(['git','branch','--show-current'], text=True).strip())

CWD: /content/Punctuation_Adder-and-Sentence-Ordering
Branch: sen-ord-1


In [5]:
# Install dependencies (Colab/Linux)
!bash scripts/colab_setup.sh

[colab_setup] Installing Python dependencies for Colab...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.8 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.2.2
    Uninstalling pip-26.2.2:
      Successfully uninstalled pip-26.2.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.4/548.4 kB 21.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 109.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [IndicTransToolkit]
[colab_setup] Done. Set HF_TOKEN for gated IndicTrans2 models if needed.


## 2) Configure Hugging Face token (for IndicTrans2 gated models)
1. Accept model access/license on Hugging Face for your chosen IndicTrans2 model (default: `ai4bharat/indictrans2-indic-en-dist-200M`).
2. In Colab, add a secret named `HF_TOKEN` (🔑 icon in left sidebar) and enable notebook access.
3. Run the cell below to load it into the environment.

In [6]:
import os

try:
    from google.colab import userdata  # type: ignore
    hf = userdata.get('HF_TOKEN')
    if hf:
        os.environ['HF_TOKEN'] = hf
        os.environ['HUGGINGFACE_HUB_TOKEN'] = hf
        print('HF_TOKEN loaded from Colab secrets.')
    else:
        print('HF_TOKEN secret not found/empty. IndicTrans2 may fall back to MarianMT if gated.')
except Exception as e:
    print('Colab userdata not available:', e)
    print('Set HF_TOKEN manually if needed: os.environ["HF_TOKEN"] = "..."')

# Translation backend selection: auto | indictrans2 | marian
os.environ.setdefault('TRANSLATION_BACKEND', 'auto')
os.environ.setdefault('INDICTRANS2_MODEL_ID', 'ai4bharat/indictrans2-indic-en-dist-200M')
print('TRANSLATION_BACKEND =', os.environ.get('TRANSLATION_BACKEND'))
print('INDICTRANS2_MODEL_ID =', os.environ.get('INDICTRANS2_MODEL_ID'))

Colab userdata not available: Secret HF_TOKEN does not exist.
Set HF_TOKEN manually if needed: os.environ["HF_TOKEN"] = "..."
TRANSLATION_BACKEND = auto
INDICTRANS2_MODEL_ID = ai4bharat/indictrans2-indic-en-dist-200M


## 3) Quick checks

In [7]:
!python setup_check.py

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
OK   data/raw/samples
OK   data/model_input
OK   data/punctuated
OK   data/ordered
OK   outputs
OK   reports
MISS models/punctuation/onnx/model_quantized.onnx

Status: MISSING ITEMS


## 4) Run pipeline stages

- **Punctuation**: ONNX CPU (`src/punctuation.py`)
- **Ordering**: mBERT pairwise scoring + Borda (`src/order_sentences.py`) — can be slow on CPU
- **Grouping + Translation**: 5-sentence groups (`src/group_and_translate.py`), IndicTrans2-preferred if available
- **Aligned outputs**: friend-style 4 files/sample (`scripts/generate_aligned_outputs.py`)

You can skip stages if artifacts already exist.

In [16]:
# Optional: regenerate punctuation (CPU)
!python src/punctuation.py

Traceback (most recent call last):
  File "/content/Punctuation_Adder-and-Sentence-Ordering/src/punctuation.py", line 13, in <module>
    runpy.run_path(str(Path(__file__).resolve().parent / "punctuation" / "restore.py"), run_name="__main__")
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen runpy>", line 294, in run_path
  File "<frozen runpy>", line 98, in _run_module_code
  File "<frozen runpy>", line 88, in _run_code
  File "/content/Punctuation_Adder-and-Sentence-Ordering/src/punctuation/restore.py", line 288, in <module>
    restorer = PunctuationRestorer()
  File "/content/Punctuation_Adder-and-Sentence-Ordering/src/punctuation/restore.py", line 48, in __init__
    raise FileNotFoundError(MODEL_FILE)
FileNotFoundError: models/punctuation/onnx/model_quantized.onnx


In [9]:
# Ordering (mBERT pairwise+Borda) - may take a while on CPU
!python src/order_sentences.py

Loading multilingual BERT for pairwise ordering: bert-base-multilingual-cased
Device=cuda batch_size=16 max_length=256 max_vram_budget_gb=1.5
CUDA device: Tesla T4
config.json: 100% 625/625 [00:00<00:00, 2.71MB/s]
tokenizer_config.json: 100% 49.0/49.0 [00:00<00:00, 273kB/s]
vocab.txt: 100% 996k/996k [00:00<00:00, 9.55MB/s]
tokenizer.json: 100% 1.96M/1.96M [00:00<00:00, 2.41MB/s]

model.safetensors: downloading bytes:  51% 364M/714M [00:02<00:01, 243MB/s, 30.9MB/s  ]
model.safetensors: reconstructing file:  36% 256M/714M [00:02<00:04, 101MB/s]
model.safetensors: downloading bytes:  74% 530M/714M [00:02<00:00, 292MB/s, 44.3MB/s  ]
model.safetensors: downloading bytes:  87% 622M/714M [00:03<00:00, 303MB/s, 50.1MB/s  ]
model.safetensors: downloading bytes:  94% 670M/714M [00:03<00:00, 276MB/s, 56.1MB/s  ]
model.safetensors: reconstructing file:  94% 670M/714M [00:04<00:00, 221MB/s, 41.1MB/s  ]
model.safetensors: downloading bytes: 100% 676M/676M [00:04<00:00, 154MB/s, 57.0MB/s  ]
model.saf

In [10]:
# Grouping + translation (IndicTrans2 if configured/available, else MarianMT fallback)
!python src/group_and_translate.py

Loading translation model (fallback MarianMT): Helsinki-NLP/opus-mt-hi-en
config.json: 100% 1.38k/1.38k [00:00<00:00, 4.01MB/s]
tokenizer_config.json: 100% 42.0/42.0 [00:00<00:00, 163kB/s]
source.spm: 100% 1.06M/1.06M [00:00<00:00, 82.6MB/s]
target.spm: 100% 813k/813k [00:00<00:00, 218MB/s]
vocab.json: 100% 2.06M/2.06M [00:00<00:00, 183MB/s]

pytorch_model.bin: downloading bytes:  20% 62.0M/304M [00:02<00:04, 50.7MB/s, 2.51MB/s  ]
pytorch_model.bin: downloading bytes:  50% 152M/304M [00:02<00:01, 132MB/s, 12.0MB/s  ] 
pytorch_model.bin: downloading bytes:  59% 181M/304M [00:02<00:01, 119MB/s, 15.5MB/s  ]
pytorch_model.bin: downloading bytes:  81% 247M/304M [00:03<00:00, 103MB/s, 20.6MB/s  ]
pytorch_model.bin: downloading bytes:  87% 263M/304M [00:03<00:00, 96.6MB/s, 21.8MB/s  ]
pytorch_model.bin: downloading bytes:  93% 284M/304M [00:03<00:00, 98.4MB/s, 22.9MB/s  ]
pytorch_model.bin: downloading bytes: 100% 285M/285M [00:04<00:00, 70.2MB/s, 23.7MB/s  ]
pytorch_model.bin: reconstructing

In [12]:
# Generate aligned outputs (outputs/sample-N.sentences_hi/en, groups_hi/en)
!python scripts/generate_aligned_outputs.py

Traceback (most recent call last):
  File "/content/Punctuation_Adder-and-Sentence-Ordering/scripts/generate_aligned_outputs.py", line 21, in <module>
    from src.group_and_translate import extract_sentences, chunked, load_translator  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ImportError: cannot import name 'extract_sentences' from 'src.group_and_translate' (/content/Punctuation_Adder-and-Sentence-Ordering/src/group_and_translate.py)


## 5) Metrics, integrity, and reports

In [13]:
!python scripts/verify_integrity.py
!python scripts/evaluate_punctuation.py
!python scripts/compute_metrics.py
!python scripts/profile_resources.py --steps prepare,integrity,metrics

sample,raw_NULL,model_input_NULL,punctuated_NULL,ordered_NULL,final_hi_NULL,raw_chars,punct_chars,ordered_chars,ordered_sentences,groups
sample-1,5,5,5,5,5,3797,3851,3852,50,10
sample-2,5,5,5,5,5,3825,3881,3882,49,10
sample-3,6,6,6,6,6,3931,3989,3990,47,10
sample-4,4,4,4,4,4,5428,5504,5505,50,10
sample-5,9,9,9,9,9,4992,5079,5080,52,11
sample-6,11,11,11,11,11,2330,2422,2423,48,10
sample-7,27,27,27,27,27,3913,4000,4001,60,12

Note: This is a quick integrity check. Always inspect outputs manually.
Wrote reports/evaluation.json and reports/run_summary.json
Ordering macro avg tau: None
Ordering macro avg pairwise acc: None
Ordering macro avg matched coverage: None
Profiling prepare...
Profiling integrity...
Profiling metrics...
Wrote /content/Punctuation_Adder-and-Sentence-Ordering/reports/resource_profile.json


In [14]:
import json, pathlib
for p in ['reports/evaluation.json','reports/run_summary.json','reports/resource_profile.json','data/final/translation_summary.json']:
    path = pathlib.Path(p)
    print('\n' + '='*80)
    print(p)
    if path.exists():
        data = json.loads(path.read_text(encoding='utf-8'))
        if p.endswith('evaluation.json'):
            print('Ordering metrics:', data.get('ordering', {}))
            print('Translation:', data.get('translation', {}))
        elif p.endswith('translation_summary.json'):
            print({k:data.get(k) for k in ['translation_backend','translation_model','group_size']})
        else:
            print(json.dumps({k:data.get(k) for k in list(data.keys())[:8]}, indent=2, ensure_ascii=False))
    else:
        print('Not found')


reports/evaluation.json
Ordering metrics: {'reference': 'data/processed/sample-*.sentences.txt (best-effort alignment by canonical text)', 'caveat': 'Metrics are computed only on matched sentences. If references are not ground-truth original order, interpret as reference-alignment rather than true ordering accuracy.', 'macro_avg_kendall_tau': None, 'macro_avg_pairwise_accuracy': None, 'macro_avg_matched_coverage': None, 'per_sample': [{'sample': 1, 'ref_sentences': 0, 'candidate_sentences': 50, 'matched_sentences': 0, 'coverage': None, 'kendall_tau_vs_ref': None, 'pairwise_accuracy_vs_ref': None}, {'sample': 2, 'ref_sentences': 0, 'candidate_sentences': 49, 'matched_sentences': 0, 'coverage': None, 'kendall_tau_vs_ref': None, 'pairwise_accuracy_vs_ref': None}, {'sample': 3, 'ref_sentences': 0, 'candidate_sentences': 47, 'matched_sentences': 0, 'coverage': None, 'kendall_tau_vs_ref': None, 'pairwise_accuracy_vs_ref': None}, {'sample': 4, 'ref_sentences': 0, 'candidate_sentences': 50, '

## 6) Inspect outputs

In [15]:
from pathlib import Path
sample = 'sample-1'
for name in [f'outputs/{sample}.sentences_hi.txt', f'outputs/{sample}.sentences_en.txt', f'outputs/{sample}.groups_hi.txt', f'outputs/{sample}.groups_en.txt']:
    p = Path(name)
    print('\n' + '='*80)
    print(name, 'exists=', p.exists())
    if p.exists():
        lines = p.read_text(encoding='utf-8').splitlines()
        for ln in lines[:6]:
            print(ln[:200])


outputs/sample-1.sentences_hi.txt exists= True
गढ़मुक्तेश्वर हिंदुओं का पावन तीर्थ है।
इसलिए हरिद्वार को गंगा द्वार भी कहते हैं।
गंगा किनारे यह भगवान शंकर की प्रसिद्ध पुरी है।
हिंदू धर्म में तीर्थ का बड़ा महत्व है।
महाराज शिव ने भगवान परशुराम से यहाँ शिव मंदिर की स्थापना कराई थी।
भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था।

outputs/sample-1.sentences_en.txt exists= True
The sanctity of the gods of the gods of Mesopotamia is the Holy House of Freeman.
So every door also calls the worldly door.
The Milky Way is the famous god's name.
Hindu religion has great importance.
His Majesty Shiva had set up the Shiva temple here from the Lord.
God's ancestors Sir Shiva lived in his van-free God only.

outputs/sample-1.groups_hi.txt exists= True
गढ़मुक्तेश्वर हिंदुओं का पावन तीर्थ है। इसलिए हरिद्वार को गंगा द्वार भी कहते हैं। गंगा किनारे यह भगवान शंकर की प्रसिद्ध पुरी है। हिंदू धर्म में तीर्थ का बड़ा महत्व है। महाराज शिव ने भगवान परशुराम से य
भगवान राम 

## 7) Download results (optional)

In [17]:
# Zip key outputs/reports for download
!zip -r results_bundle.zip outputs reports data/final data/ordered data/punctuated README.md

try:
    from google.colab import files  # type: ignore
    files.download('results_bundle.zip')
except Exception as e:
    print('Download manually from file browser:', e)

  adding: outputs/ (stored 0%)
  adding: outputs/sample-4.sentences_hi.txt (deflated 75%)
  adding: outputs/sample-7.groups_en.txt (deflated 51%)
  adding: outputs/sample-5.sentences_hi.txt (deflated 77%)
  adding: outputs/sample-7.sentences_hi.txt (deflated 73%)
  adding: outputs/sample-3.sentences_hi.txt (deflated 78%)
  adding: outputs/sample-3.sentences_en.txt (deflated 52%)
  adding: outputs/sample-5.groups_en.txt (deflated 53%)
  adding: outputs/sample-1.groups_hi.txt (deflated 77%)
  adding: outputs/sample-4.groups_hi.txt (deflated 75%)
  adding: outputs/sample-5.sentences_en.txt (deflated 54%)
  adding: outputs/sample-3.groups_en.txt (deflated 52%)
  adding: outputs/sample-7.sentences_en.txt (deflated 51%)
  adding: outputs/sample-6.groups_hi.txt (deflated 74%)
  adding: outputs/sample-5.groups_hi.txt (deflated 77%)
  adding: outputs/sample-6.sentences_en.txt (deflated 50%)
  adding: outputs/sample-2.groups_hi.txt (deflated 77%)
  adding: outputs/sample-4.groups_en.txt (deflate

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notes
- **NULL markers (67 total)** are preserved; review `scripts/null_diagnostic.py` and `reports/null_punct_analysis.json`.
- **Ordering/translation quality** must be manually reviewed; metrics in reports are partial/proxy where noted.
- If IndicTrans2 is used, record model ID, environment, and any token/access steps for reproducibility (do not share tokens).
- For GPU runs, measure peak VRAM and ensure it stays < 1.5 GB per assignment constraint.